# Пайплайн аугментации

Ноутбук последовательно запускает все этапы аугментации несбалансированного датасета
и оценивает результат через baseline-классификаторы.

**Этапы аугментации:**
1. LLM-генерация (< 15 → 15)
2. Парафраз через LLM (< 35 → 35)
3. Обратный перевод (< 50 → 50)

**Классификация (baseline):**
- Linear SVM
- Logistic Regression
- Gaussian Naive Bayes
- Random Forest

___
## Подготовка окружения

### Локальный запуск


In [ ]:
import subprocess
import sys
from pathlib import Path
import pandas as pd

# Корень проекта — AUG/
PROJECT_ROOT = Path("/Users/kvt/Documents/VKR/code/AUG")

# Добавляем в sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
os.chdir(PROJECT_ROOT)

# Установка зависимостей
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements.txt")])

from src.utils.data_loader import load_dataset, get_class_distribution, LABEL_COL, RANDOM_SEED
from src.classification.evaluate import evaluate_model

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Папка данных:   {PROJECT_ROOT.parent / 'Data'}")

Корень проекта: /Users/kvt/Documents/VKR/code/AUG
Папка данных:   /Users/kvt/Documents/VKR/code/Data


FileNotFoundError: [Errno 2] No such file or directory: '/Users/kvt/Documents/VKR/code/AUG'

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import sys
from pathlib import Path
import pandas as pd

# Корень проекта на Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/VKR/code/AUG")

# Добавляем корень в sys.path, чтобы работали импорты вида from src.utils...
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Переходим в корень проекта — чтобы относительные пути в модулях работали корректно
%cd {PROJECT_ROOT}

from src.utils.data_loader import load_dataset, get_class_distribution, LABEL_COL

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Папка данных:   {PROJECT_ROOT.parent / 'Data'}")

/content/drive/MyDrive/VKR/code/AUG
Корень проекта: /content/drive/MyDrive/VKR/code/AUG
Папка данных:   /content/drive/MyDrive/VKR/code/Data


In [3]:
# Ставим зависимости из requirements.txt
!pip install -q -r {PROJECT_ROOT}/requirements.txt

KeyboardInterrupt: 

___
## Исходные данные

In [5]:
# Загружаем исходный датасет (после EDA)
df_original = load_dataset(stage=0)

print(f"\nВсего записей: {len(df_original)}")
print(f"Классов: {df_original[LABEL_COL].nunique()}")
print(f"\nРаспределение по классам:")
dist = get_class_distribution(df_original)
for name, count in dist.items():
    print(f"  {name}: {count}")

[Данные] Найден чекпоинт этапа 0: data_after_eda.csv (1755 записей)

Всего записей: 1755
Классов: 36

Распределение по классам:
  Блок технического директора: 246
  Блок директора по мощностям: 241
  Блок директора по строительству: 165
  Управление по проектным работам: 135
  Блок заместителя генерального директора по безопасности: 123
  Генеральный директор: 102
  Проект "Нефтяные краюшки": 79
  Блок деректора по газу: 71
  Блок заместителя генерального директора по закупкам: 67
  Блок заместителя генерального директора по организационным вопросам: 56
  Проект сервиса скважин: 45
  Блок директора по проектированию: 43
  Проект "Новая нефть": 38
  Проект "Северная деревня": 36
  Блок операционного директора: 33
  Блок директора по газовым проектам: 31
  Блок заместителя генерального директора по защите: 26
  Блок финансового директора: 25
  Блок директора по портфелю: 24
  Управление землеустроительных работ: 23
  Проект "Трубопроводный транспорт Главного НГКМ": 20
  Проект "Восточный

___
## Этап 1: LLM-генерация (< 15 -> 15)

Классы с менее чем 15 примерами дополняются новыми текстами,
сгенерированными через LLM.

In [5]:
# Путь до конфига модели 
# CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen.json")
# урезанная
# CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen_3b.json")
# квантизированная
CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen_14b_unsloth.json")
print(f"Конфиг модели: {CONFIG_PATH}")

Конфиг модели: /content/drive/MyDrive/VKR/code/configs/model_qwen_14b_unsloth.json


In [20]:
from src.augmentation.stage1_llm_generate import run as run_stage1

run_stage1(CONFIG_PATH)

ЭТАП 1: LLM-генерация (< 15 → 15)
[Данные] Чекпоинт этапа 1 не найден, загружен этап 0: data_after_eda.csv (1755 записей)

[Этап 1] Классов для аугментации: 11
  «Имущественные вопросы»: 2 → нужно ещё 13
  «Подразделение по информационным технологиям»: 2 → нужно ещё 13
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 2 → нужно ещё 13
  «Блок заместителя генерального директора по строительству»: 2 → нужно ещё 13
  «Проект «Обустройство объектов Новейшей нейти»»: 3 → нужно ещё 12
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 5 → нужно ещё 10
  «Проект "Обустройство площадных объектов НГКМ Поменбше"»: 6 → нужно ещё 9
  «Управление коммуникаций»: 7 → нужно ещё 8
  «Проект «Обустройство Интересного лицензионного участка»»: 7 → нужно ещё 8
  «Блок бизнес-директора»: 8 → нужно ещё 7
  «Проект "Южный"»: 12 → нужно ещё 3
[LLM] Загружаю модель: unsloth/Qwen2.5-14B-Instruct-bnb-4bit


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект "Южный"»: есть 12, нужно ещё 3
  [Попытка 1/10] Нужно ещё 3 текстов
  [Парсинг] Отброшено обрезанное письмо: «Нефтепромышленное предприятие [ORGANIZATION]  
Центр безопас...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект "Южный"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект "Южный"»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 2 текстов
  [Попытка 2] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект "Южный"»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект "Южный"»: прошло 2/2, отсеяно 0
[Этап 1] Класс «Проект "Южный"»: добавлено 3 текстов
Пример сгенерированного письма:
--------------------------------------------------
Нефтегазовый концерн [ORGANIZATION]  
Технический отдел  
Адрес: [LOCATION], улица [STREET_NAME], д. [BUILDING_NUMBER], офис [OFFICE_NUMBER]; Тел.: [PHONE_NUMBER]; E-mail: [EMAIL_ADDRESS]; ОКПО [CODE], ОГРН [REGISTRATION_

In [6]:
import torch
print(f"Память GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} ГБ")

Память GPU: 0.0 ГБ


In [22]:
df_after_s1 = load_dataset(stage=1)
dist_s1 = get_class_distribution(df_after_s1)
while (dist_s1 < 15).sum() != 0:# Проверяем результат после этапа 1
    print(f"Записей после этапа 1: {len(df_after_s1)}")
    print(f"Классов с < 15 примерами: {(dist_s1 < 15).sum()}")
    print(f"{"="*100}\n\n")
    print("Повторяем 1й этап")
    print(f"\n\n{"="*100}")
    run_stage1(CONFIG_PATH)
    df_after_s1 = load_dataset(stage=1)
    dist_s1 = get_class_distribution(df_after_s1)
print("Этап 1: Генерация с помощью LLM полностью завершен.")

[Данные] Найден чекпоинт этапа 1: data_after_stage1.csv (1832 записей)
Записей после этапа 1: 1832
Классов с < 15 примерами: 6


Повторяем 1й этап


ЭТАП 1: LLM-генерация (< 15 → 15)
[Данные] Найден чекпоинт этапа 1: data_after_stage1.csv (1832 записей)

[Этап 1] Классов для аугментации: 6
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 2 → нужно ещё 13
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 7 → нужно ещё 8
  «Управление коммуникаций»: 10 → нужно ещё 5
  «Подразделение по информационным технологиям»: 11 → нужно ещё 4
  «Проект «Обустройство объектов Новейшей нейти»»: 14 → нужно ещё 1
  «Проект «Обустройство Интересного лицензионного участка»»: 14 → нужно ещё 1
[LLM] Загружаю модель: unsloth/Qwen2.5-14B-Instruct-bnb-4bit


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Обустройство объектов Новейшей нейти»»: есть 14, нужно ещё 1
  [Попытка 1/10] Нужно ещё 1 текстов
  [Попытка 1] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: прошло 2/2, отсеяно 0
[Этап 1] Класс «Проект «Обустройство объектов Новейшей нейти»»: добавлено 1 текстов
Пример сгенерированного письма:
--------------------------------------------------
[DATE_TIME] исх. No [DOCUMENT_NUMBER] Руководителю проекта [OBJECT_NAME] [ORGANIZATION] [PERSON_FIO]

Уважаемый [PERSON_FIO],

В рамках выполнения договора подряда №[DOCUMENT_NUMBER] на проведение строительно-монтажных работ по объекту [OBJECT_NAME], прошу Вас рассмотреть вопрос о согласовании перечня необходимых строительных материалов и оборудования для продолжения работ на данном этапе.

Наши специалисты подготовили 

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: есть 12, нужно ещё 3
  [Попытка 1/10] Нужно ещё 3 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "Альтернатива", Москва, улица Ленина, дом 15**
Тел.: +...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 2 текстов
  [Парсинг] Отброшено обрезанное письмо: «ВРИО генерального директора ПАО [ORGANIZATION] [PERSON],
Зам...»
  [Попытка 2] Распарсено 3 текстов из ответа модели
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: проверяем 3 сгенерированных текстов
  [Длина] Класс «Блок исполнительного директора по реализаци

...

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 4, нужно ещё 11
  [Попытка 1/10] Нужно ещё 11 текстов
  [Парсинг] Отброшено обрезанное письмо: «**О...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 10 текстов
  [Парсинг] Отброшено обрезанное письмо: «**АО "Нефтетехника"**

Конт...»
  [Попытка 2] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 3/10] Нужно ещё 9 текстов
  [Попытка 3] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: про

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 6, нужно ещё 9
  [Попытка 1/10] Нужно ещё 9 текстов
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "ТехноГазстрой"**

Контактная информация:
Адрес: г. Са...»
  [Попытка 2] Распарсено 0 текстов из ответа модели
  [Попытка 3/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «енные ограничения на проезд техники из-за весеннего паводка ...»
  [Попытка 3] Распарсено 0 текстов из ответа модели
  [Попытка 4/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «这些例子遵循了您提供的格式，并且是虚构的内容。如果您需要更具体的信息或有其他需求，请告诉我！...»
  [Попытка 4] Распарсено 1 текстов из ответа модели
[Вали

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 13, нужно ещё 2
  [Попытка 1/10] Нужно ещё 2 текстов
  [Парсинг] Отброшено обрезанное письмо: «Уважаемый Петр Сергеевич,

По результатам анализа возможност...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 1 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "НеваНефть"**

Контактная информация:  
Адрес: г. Санк...»
  [Попытка 2] Распарсено 0 текстов из ответа модели
  [Попытка 3/10] Нужно ещё 1 текстов
  [Попытка 3] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 2

## 3. Этап 2: Парафраз через LLM (< 35 -> 35)

Классы с менее чем 35 примерами пополняются через перефразирование
существующих текстов — LLM получает оригинал и переписывает его
другими словами, сохраняя смысл.

In [11]:
from src.augmentation.stage2_paraphrase import run as run_stage2

run_stage2(CONFIG_PATH)

ЭТАП 2: Парафраз через LLM (< 35 → 35)
[Данные] Найден чекпоинт этапа 2: data_after_stage2.csv (2216 записей)

[Этап 2] Классов для аугментации: 1
  «Проект «Обустройство объектов Новейшей нейти»»: 34 → нужно ещё 1
[LLM] Загружаю модель: unsloth/Qwen2.5-14B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[LLM] Модель загружена, устройство: cuda:0

[Этап 2] Класс «Проект «Обустройство объектов Новейшей нейти»»: есть 34, нужно ещё 1
  [Попытка 1/5] Нужно ещё 1 парафразов
    Перефразировано 1/1
  [Попытка 1] Получено 1 парафразов
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: проверяем 1 сгенерированных текстов
  [Сходство] Класс «Проект «Обустройство объектов Новейшей нейти»»: отсеяно 1 текстов (косинусное сходство > 0.95)
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: прошло 0/1, отсеяно 1
  [Попытка 2/5] Нужно ещё 1 парафразов
    Перефразировано 1/1
  [Попытка 2] Получено 1 парафразов
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: проверяем 1 сгенерированных текстов
  [Сходство] Класс «Проект «Обустройство объектов Новейшей нейти»»: отсеяно 1 текстов (косинусное сходство > 0.95)
[Валидация] Класс «Проект «Обустройство объ

In [5]:
# Проверяем результат после этапа 2
df_after_s2 = load_dataset(stage=2)
dist_s2 = get_class_distribution(df_after_s2)

print(f"Записей после этапа 2: {len(df_after_s2)}")
print(f"Классов с < 35 примерами: {(dist_s2 < 35).sum()}")

[Данные] Найден чекпоинт этапа 2: data_after_stage2.csv (2217 записей)
Записей после этапа 2: 2217
Классов с < 35 примерами: 0


## 4. Этап 3: Обратный перевод (< 50 -> 50)

Оставшиеся классы с менее чем 50 примерами дополняются через
обратный перевод RU → EN → RU (facebook/nllb-200-distilled-600M).

In [7]:
from src.augmentation.stage3_back_translation import run as run_stage3

run_stage3()

ЭТАП 3: Обратный перевод (< 50 → 50)
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2566 записей)

[Этап 3] Классов для аугментации: 13
  «Проект "Восточный"»: 48 → нужно ещё 2
  «Проект "Северная деревня"»: 48 → нужно ещё 2
  «Блок бизнес-директора»: 48 → нужно ещё 2
  «Проект "Обустройство площадных объектов НГКМ Поменбше"»: 48 → нужно ещё 2
  «Проект «Обустройство Интересного лицензионного участка»»: 48 → нужно ещё 2
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 48 → нужно ещё 2
  «Проект "Южный"»: 49 → нужно ещё 1
  «Проект «Обустройство объектов Новой нефти»»: 49 → нужно ещё 1
  «Проект "Трубопроводный транспорт Главного НГКМ"»: 49 → нужно ещё 1
  «Блок директора по газовым проектам»: 49 → нужно ещё 1
  «Блок финансового директора»: 49 → нужно ещё 1
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 49 → нужно ещё 1
  «Подразделение по информационным технологиям»: 49 → нужно ещё 1
[Перевод] Загружаю NLLB-200: facebook/nllb-200-dist

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[Перевод] Модель загружена

[Этап 3] Попытка 1/5: 13 классов ещё не доведены до цели
  Всего источников для перевода: 19


[Валидация] Класс «Проект "Южный"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект "Южный"»: прошло 1/1, отсеяно 0
  [1] «Проект "Южный"»: получено 1, валидных 1, принято 1, итого 1/1
[Валидация] Класс «Проект «Обустройство объектов Новой нефти»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Обустройство объектов Новой нефти»»: прошло 1/1, отсеяно 0
  [1] «Проект «Обустройство объектов Новой нефти»»: получено 1, валидных 1, принято 1, итого 1/1
[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: проверяем 1 сгенерированных текстов
  [Сходство] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: отсеяно 1 текстов (косинусное сходство > 0.95)
[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: прошло 0/1, отсеяно 1
  [1] «Проект "Трубопроводный транспорт Главного НГКМ"»: получено 1, валидных 0, принято 0, итого 0/1
[Данные] Сохранён чекпоинт этапа 3: data_after_stage3.csv (2568 записей)
  [Чекпоинт] Сохранено 2568 за

[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: проверяем 1 сгенерированных текстов
  [Дубликаты] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: удалено 1 точных дубликатов
[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: прошло 0/1, отсеяно 1
  [2] «Проект "Трубопроводный транспорт Главного НГКМ"»: получено 1, валидных 0, принято 0, итого 0/1
[Данные] Сохранён чекпоинт этапа 3: data_after_stage3.csv (2576 записей)
  [Чекпоинт] Сохранено 2576 записей после 3 классов
[Валидация] Класс «Блок директора по газовым проектам»: проверяем 1 сгенерированных текстов
  [Дубликаты] Класс «Блок директора по газовым проектам»: удалено 1 точных дубликатов
[Валидация] Класс «Блок директора по газовым проектам»: прошло 0/1, отсеяно 1
  [2] «Блок директора по газовым проектам»: получено 1, валидных 0, принято 0, итого 0/1
[Валидация] Класс «Блок финансового директора»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Блок финансового директора»: п

[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект "Трубопроводный транспорт Главного НГКМ"»: прошло 1/1, отсеяно 0
  [3] «Проект "Трубопроводный транспорт Главного НГКМ"»: получено 1, валидных 1, принято 1, итого 1/1
[Данные] Сохранён чекпоинт этапа 3: data_after_stage3.csv (2579 записей)
  [Чекпоинт] Сохранено 2579 записей после 3 классов
[Валидация] Класс «Блок директора по газовым проектам»: проверяем 1 сгенерированных текстов
  [Язык] Класс «Блок директора по газовым проектам»: отсеяно 1 текстов (не русский язык)
[Валидация] Класс «Блок директора по газовым проектам»: прошло 0/1, отсеяно 1
  [3] «Блок директора по газовым проектам»: получено 1, валидных 0, принято 0, итого 0/1
[Валидация] Класс «Проект "Восточный"»: проверяем 1 сгенерированных текстов
  [Сходство] Класс «Проект "Восточный"»: отсеяно 1 текстов (косинусное сходство > 0.95)
[Валидация] Класс «Проект "Восточный"»: прошло 0/1, отсеяно 1
  [

[Валидация] Класс «Блок директора по газовым проектам»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Блок директора по газовым проектам»: прошло 1/1, отсеяно 0
  [4] «Блок директора по газовым проектам»: получено 1, валидных 1, принято 1, итого 1/1
[Валидация] Класс «Проект "Восточный"»: проверяем 1 сгенерированных текстов
  [Язык] Класс «Проект "Восточный"»: отсеяно 1 текстов (не русский язык)
[Валидация] Класс «Проект "Восточный"»: прошло 0/1, отсеяно 1
  [4] «Проект "Восточный"»: получено 1, валидных 0, принято 0, итого 1/2
[Валидация] Класс «Блок бизнес-директора»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Блок бизнес-директора»: прошло 1/1, отсеяно 0
  [4] «Блок бизнес-директора»: получено 1, валидных 1, принято 1, итого 2/2
[Валидация] Класс «Проект "Обустройство площадных объектов НГКМ Поменбше"»: проверяем 1 сгенерированных текстов
  [Дубликаты] Класс «Проект "Обустройство площадных объектов НГКМ Поменбше"»: удалено 1 точных дубликатов
[Валидация] Класс «

[Валидация] Класс «Проект "Восточный"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект "Восточный"»: прошло 1/1, отсеяно 0
  [5] «Проект "Восточный"»: получено 1, валидных 1, принято 1, итого 2/2
[Валидация] Класс «Проект "Обустройство площадных объектов НГКМ Поменбше"»: проверяем 1 сгенерированных текстов
  [Дубликаты] Класс «Проект "Обустройство площадных объектов НГКМ Поменбше"»: удалено 1 точных дубликатов
[Валидация] Класс «Проект "Обустройство площадных объектов НГКМ Поменбше"»: прошло 0/1, отсеяно 1
  [5] «Проект "Обустройство площадных объектов НГКМ Поменбше"»: получено 1, валидных 0, принято 0, итого 1/2
[Этап 3] Класс «Проект "Южный"»: добавлено 1 текстов
Пример сгенерированного письма:
--------------------------------------------------
В ответ на ваш запрос No. [ДОКУМЕНТНОМ_НОМЕР] от [ТИМ], мы предлагаем следующие характеристики нашей организации: *условное внутреннее диаметры [ID], *условное диаметры выбросов [ID], *операционный давление [IDP] атмосферы, *тем

In [8]:
# Проверяем финальный результат
df_final = load_dataset(stage=3)
dist_final = get_class_distribution(df_final)

print(f"Записей после всех этапов: {len(df_final)}")
print(f"Классов с < 50 примерами: {(dist_final < 50).sum()}")
print(f"\nМинимум примеров в классе: {dist_final.min()}")
print(f"Максимум примеров в классе: {dist_final.max()}")

[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2584 записей)
Записей после всех этапов: 2584
Классов с < 50 примерами: 1

Минимум примеров в классе: 49
Максимум примеров в классе: 246


In [ ]:
import matplotlib.pyplot as plt

# Сравниваем распределение до и после аугментации
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

dist.plot(kind="bar", ax=axes[0], color="salmon")
axes[0].set_title("До аугментации")
axes[0].set_ylabel("Количество примеров")
axes[0].tick_params(axis="x", rotation=45)

dist_final.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("После аугментации")
axes[1].set_ylabel("Количество примеров")
axes[1].axhline(y=50, color="g", linestyle="--", alpha=0.5, label="Целевой минимум (50)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Классификация — оценка качества аугментации

Запускаем baseline-модели на финальном датасете.
Все используют SBERT-эмбеддинги (ai-forever/sbert_large_nlu_ru)
и StratifiedKFold кросс-валидацию (k=5).

Общая логика вынесена в `evaluate.py` — каждый классификатор
только задаёт модель и сетку параметров.

### 5.1 Linear SVM

In [2]:
from sklearn.svm import LinearSVC
from src.classification.evaluate import evaluate_model
from src.utils.data_loader import RANDOM_SEED

In [4]:
from sklearn.svm import LinearSVC
from src.classification.evaluate import evaluate_model
from src.utils.data_loader import RANDOM_SEED

evaluate_model(
    name="Linear SVM",
    estimator=LinearSVC(max_iter=10000, random_state=RANDOM_SEED, dual="auto"),
    param_grid={"C": [0.01, 0.1, 1, 10]},
)

КЛАССИФИКАЦИЯ: Linear SVM
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2584 записей)
[Эмбеддинги] Загружаю модель: ai-forever/sbert_large_nlu_ru


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

[Эмбеддинги] Модель загружена, размерность: 1024
[Эмбеддинги] Загружено из кэша ((2584, 1024))

[Linear SVM] Датасет: 2584 примеров, 1024 фичей, 36 классов

[Linear SVM] GridSearchCV (5 фолдов), параметры: {'C': [0.01, 0.1, 1, 10]}
[Linear SVM] Лучшие параметры: {'C': 10} (macro F1 = 0.5987)
  {'C': 0.01} → macro F1 = 0.1283
  {'C': 0.1} → macro F1 = 0.3442
  {'C': 1} → macro F1 = 0.5321
  {'C': 10} → macro F1 = 0.5987

Метрика                 Среднее        Std
------------------------------------------
  Accuracy               0.6049     0.0166
  Macro F1               0.5987     0.0159
  Weighted F1            0.5960     0.0160

[Linear SVM] Classification report (cross-validated):
                                                                              precision    recall  f1-score   support

                                                       Блок бизнес-директора       0.77      0.74      0.76        50
                                                      Блок деректора 

### 5.2 Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression

evaluate_model(
    name="Logistic Regression",
    estimator=LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_SEED,
    ),
    param_grid={"C": [0.01, 0.1, 1, 10]},
)

КЛАССИФИКАЦИЯ: Logistic Regression
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2584 записей)
[Эмбеддинги] Загружаю модель: ai-forever/sbert_large_nlu_ru


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Эмбеддинги] Модель загружена, размерность: 1024
[Эмбеддинги] Загружено из кэша ((2584, 1024))

[Logistic Regression] Датасет: 2584 примеров, 1024 фичей, 36 классов

[Logistic Regression] GridSearchCV (5 фолдов), параметры: {'C': [0.01, 0.1, 1, 10]}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[Logistic Regression] Лучшие параметры: {'C': 10} (macro F1 = 0.5243)
  {'C': 0.01} → macro F1 = 0.0146
  {'C': 0.1} → macro F1 = 0.0145
  {'C': 1} → macro F1 = 0.2600
  {'C': 10} → macro F1 = 0.5243

Метрика                 Среднее        Std
------------------------------------------
  Accuracy               0.5403     0.0120
  Macro F1               0.5243     0.0126
  Weighted F1            0.5253     0.0116

[Logistic Regression] Classification report (cross-validated):
                                                                              precision    recall  f1-score   support

                                                       Блок бизнес-директора       0.62      0.68      0.65        50
                                                      Блок деректора по газу       0.29      0.15      0.20        71
                                          Блок директора по газовым проектам       0.50      0.34      0.40        50
                                               

### 5.3 Gaussian Naive Bayes

In [6]:
from sklearn.naive_bayes import GaussianNB

evaluate_model(
    name="Gaussian Naive Bayes",
    estimator=GaussianNB(),
)

КЛАССИФИКАЦИЯ: Gaussian Naive Bayes
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2584 записей)
[Эмбеддинги] Загружаю модель: ai-forever/sbert_large_nlu_ru


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Эмбеддинги] Модель загружена, размерность: 1024
[Эмбеддинги] Загружено из кэша ((2584, 1024))

[Gaussian Naive Bayes] Датасет: 2584 примеров, 1024 фичей, 36 классов

Метрика                 Среднее        Std
------------------------------------------
  Accuracy               0.4110     0.0114
  Macro F1               0.4130     0.0083
  Weighted F1            0.4076     0.0114

[Gaussian Naive Bayes] Classification report (cross-validated):
                                                                              precision    recall  f1-score   support

                                                       Блок бизнес-директора       0.47      0.68      0.55        50
                                                      Блок деректора по газу       0.22      0.34      0.27        71
                                          Блок директора по газовым проектам       0.43      0.30      0.35        50
                                                 Блок директора по мощностям    

### 5.4 Random Forest

In [3]:
from sklearn.ensemble import RandomForestClassifier

evaluate_model(
    name="Random Forest",
    estimator=RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    param_grid={"n_estimators": [100, 300, 500], "max_depth": [None, 30, 50]},
)

КЛАССИФИКАЦИЯ: Random Forest
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2584 записей)
[Эмбеддинги] Загружаю модель: ai-forever/sbert_large_nlu_ru


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Эмбеддинги] Модель загружена, размерность: 1024
[Эмбеддинги] Загружено из кэша ((2584, 1024))

[Random Forest] Датасет: 2584 примеров, 1024 фичей, 36 классов

[Random Forest] GridSearchCV (5 фолдов), параметры: {'n_estimators': [100, 300, 500], 'max_depth': [None, 30, 50]}
[Random Forest] Лучшие параметры: {'max_depth': 50, 'n_estimators': 500} (macro F1 = 0.4442)
  {'max_depth': None, 'n_estimators': 100} → macro F1 = 0.3966
  {'max_depth': None, 'n_estimators': 300} → macro F1 = 0.4377
  {'max_depth': None, 'n_estimators': 500} → macro F1 = 0.4436
  {'max_depth': 30, 'n_estimators': 100} → macro F1 = 0.4027
  {'max_depth': 30, 'n_estimators': 300} → macro F1 = 0.4357
  {'max_depth': 30, 'n_estimators': 500} → macro F1 = 0.4361
  {'max_depth': 50, 'n_estimators': 100} → macro F1 = 0.3966
  {'max_depth': 50, 'n_estimators': 300} → macro F1 = 0.4386
  {'max_depth': 50, 'n_estimators': 500} → macro F1 = 0.4442

Метрика                 Среднее        Std
---------------------------------